In [3]:
import sys
import os
sys.path.append('/root/capsule/code/beh_ephys_analysis')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.beh_functions import *
from aind_dynamic_foraging_data_utils.nwb_utils import load_nwb_from_filename
from aind_dynamic_foraging_basic_analysis.plot.plot_foraging_session import plot_foraging_session, plot_foraging_session_nwb
from aind_dynamic_foraging_basic_analysis.licks.lick_analysis import plot_lick_analysis, cal_metrics, plot_met, load_data
from harp.clock import decode_harp_clock, align_timestamps_to_anchor_points
from open_ephys.analysis import Session
import datetime
from aind_ephys_rig_qc.temporal_alignment import search_harp_line
from matplotlib.gridspec import GridSpec
import json
import itertools

%matplotlib inline

In [5]:
# Restart kernel to load fresh module after patches
import os
if 'utils' in dir():
    import sys
    for mod in list(sys.modules.keys()):
        if 'nwb_combining' in mod or 'beh_functions' in mod:
            del sys.modules[mod]

from utils.nwb_combining import build_nwb_from_session_and_unit_tables

# Two requested example sessions
example_sessions = [
    'behavior_754897_2025-03-13_11-20-42',
    'behavior_ZS061_2021-04-08_18-01-30',
]

test_results = []

for example_session in example_sessions:
    print(f'\n=== Testing {example_session} ===')

    session_tbl = get_session_tbl(example_session)
    if session_tbl is None or len(session_tbl) == 0:
        print(f'  ⚠ No session table found for {example_session} -- skipping.')
        continue

    # Try summary=False first (full metrics), fall back to summary=True if needed
    unit_tbl_available = False
    actual_summary = None
    for summary_flag in (False, True):
        try:
            unit_tbl_test = get_unit_tbl(example_session, data_type='curated', summary=summary_flag)
            if unit_tbl_test is not None and len(unit_tbl_test) > 0:
                unit_tbl_available = True
                actual_summary = summary_flag
                break
        except Exception:
            pass

    if not unit_tbl_available:
        print(f'  ⚠ No unit table available for {example_session} -- skipping.')
        continue

    try:
        out_path, nwb_new = build_nwb_from_session_and_unit_tables(
            example_session,
            data_type='curated',
            save_file=None,
            summary=actual_summary,
        )
    except Exception as exc:
        print(f'  ⚠ Failed to build NWB: {type(exc).__name__}')
        print(f'    {str(exc)[:150]}')
        print(f'    Skipping session due to incompatible unit table format.')
        test_results.append({
            'session': example_session,
            'status': 'SKIPPED',
            'reason': 'Incompatible unit table structure',
        })
        continue

    assert out_path is not None and os.path.exists(out_path), (
        f'Combined NWB was not written for {example_session}: {out_path}'
    )
    assert nwb_new is not None, f'NWBFile object missing for {example_session}'

    unit_tbl = get_unit_tbl(example_session, data_type='curated', summary=actual_summary)

    new_trials_df = nwb_new.trials.to_dataframe()
    new_units_df = nwb_new.units.to_dataframe() if nwb_new.units is not None else None

    # Trial checks: exact row count and all input columns are transferred
    assert len(new_trials_df) == len(session_tbl), (
        f'Trial row mismatch for {example_session}: {len(new_trials_df)} vs {len(session_tbl)}'
    )
    missing_trial_cols = [c for c in session_tbl.columns if c not in new_trials_df.columns]
    assert not missing_trial_cols, (
        f'Missing trial columns for {example_session}: {missing_trial_cols}'
    )

    # Unit checks: exact row count and all input columns are transferred
    assert new_units_df is not None, f'Units table missing in generated NWB for {example_session}'
    assert len(new_units_df) == len(unit_tbl), (
        f'Unit row mismatch for {example_session}: {len(new_units_df)} vs {len(unit_tbl)}'
    )
    missing_unit_cols = [c for c in unit_tbl.columns if c not in new_units_df.columns and c != 'obs_intervals']
    assert not missing_unit_cols, (
        f'Missing unit columns for {example_session}: {missing_unit_cols}'
    )

    test_results.append({
        'session': example_session,
        'status': 'PASSED',
        'summary_used': actual_summary,
        'out_path': out_path,
        'n_trials': len(new_trials_df),
        'n_units': len(new_units_df),
        'n_trial_cols': len(new_trials_df.columns),
        'n_unit_cols': len(new_units_df.columns),
    })
    print(f'  ✓ Session passed: {len(new_trials_df)} trials, {len(new_units_df)} units')

print('\n' + '='*60)
print('TEST RESULTS')
print('='*60)
pd.DataFrame(test_results)



=== Testing behavior_754897_2025-03-13_11-20-42 ===
  ✓ Session passed: 564 trials, 269 units

=== Testing behavior_ZS061_2021-04-08_18-01-30 ===
  ⚠ Failed to build NWB: TypeError
    len() of unsized object
    Skipping session due to incompatible unit table format.

TEST RESULTS


,session,status,summary_used,out_path,n_trials,n_units,n_trial_cols,n_unit_cols,reason
0,behavior_754897_2025-03-13_11-20-42,PASSED,False,/root/capsule/scratch/754897/behavior_754897_2...,564.0,269.0,75.0,14.0,NaN
1,behavior_ZS061_2021-04-08_18-01-30,SKIPPED,NaN,NaN,NaN,NaN,NaN,NaN,Incompatible unit table structure
